In [150]:
import torch
import torch.nn as nn
import torch.optim as optim

In [152]:
# Simple Vocabulary
sentences = [
    ("the cat sits on the mat", "the mat is soft", True),
    ("the dog runs in the park", "birds fly in the sky", False),
    ("the sun rises in the east", "it is very bright", True),
    ("fish swim in the water", "the moon shines at night", False),
]

In [153]:
#  Tokenization + Vocabulary
all_words = set()
for sent_a, sent_b, _ in sentences:
    all_words.update(sent_a.split())
    all_words.update(sent_b.split())

word2idx = {w: i for i, w in enumerate(sorted(all_words))}
vocab_size = len(word2idx)

In [154]:
# Masking function for MLM
def mask_sentence(tokens, mask_idx=-1):
    masked_tokens = []
    labels = []
    for t in tokens:
        if torch.rand(1).item() < 0.15:  # 15% chance to mask
            masked_tokens.append("[MASK]")
            labels.append(t)
        else:
            masked_tokens.append(t)
            labels.append(None)
    return masked_tokens, labels


In [155]:
# Mini-BERT Model (MLM + NSP)

class MiniBERT(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=2)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc_mlm = nn.Linear(d_model, vocab_size)  # For MLM
        self.fc_nsp = nn.Linear(d_model, 2)           # For NSP (IsNext / NotNext)

    def forward(self, x):
        emb = self.embedding(x)              # [seq_len, batch_size, d_model]
        out = self.transformer(emb)          # [seq_len, batch_size, d_model]

        mlm_logits = self.fc_mlm(out)        # [seq_len, batch_size, vocab_size]

        cls_output = out[0, :, :]            # CLS token for all batch -> [batch_size, d_model]
        nsp_logits = self.fc_nsp(cls_output) # [batch_size, 2]

        return mlm_logits, nsp_logits

In [156]:
# Training Setup
model = MiniBERT(vocab_size)
criterion_mlm = nn.CrossEntropyLoss(ignore_index=-1)
criterion_nsp = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [157]:
# Training Loop
# -------------------------
for epoch in range(50):
    total_loss = 0
    for (sent_a, sent_b, is_next) in sentences:
        # MLM: mask sentence B
        masked_b, labels = mask_sentence(sent_b.split())
        tokens = ["[CLS]"] + sent_a.split() + ["[SEP]"] + masked_b + ["[SEP]"]
        label_ids = [-1]*(len(sent_a.split())+2) + [word2idx[w] if w else -1 for w in labels] + [-1]

        input_ids = [word2idx.get(tok, vocab_size) for tok in tokens]
        x = torch.tensor(input_ids).unsqueeze(1)
        y_mlm = torch.tensor(label_ids).unsqueeze(1)
        y_nsp = torch.tensor([1 if is_next else 0], dtype=torch.long)
        optimizer.zero_grad()
        mlm_logits, nsp_logits = model(x)

        # MLM loss
        loss_mlm = criterion_mlm(mlm_logits.view(-1, vocab_size), y_mlm.view(-1))

        # NSP loss
        loss_nsp = criterion_nsp(nsp_logits, y_nsp)

        # Total loss
        loss = loss_mlm + loss_nsp
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Total Loss: {total_loss:.4f}")

print("Training finished!")

Epoch 0, Total Loss: nan
Epoch 10, Total Loss: nan
Epoch 20, Total Loss: nan
Epoch 30, Total Loss: nan
Epoch 40, Total Loss: nan
Training finished!


In [158]:
# Test MLM + NSP
model.eval()
with torch.no_grad():
    for sent_a, sent_b, is_next in sentences:
        # Mask some words in sent_b for MLM
        masked_b, labels = mask_sentence(sent_b.split())
        tokens = ["[CLS]"] + sent_a.split() + ["[SEP]"] + masked_b + ["[SEP]"]

        # Convert to indices
        input_ids = [word2idx.get(tok, vocab_size) for tok in tokens]  # unknown words -> vocab_size
        x = torch.tensor(input_ids).unsqueeze(1)

        # Forward pass
        mlm_logits, nsp_logits = model(x)

        # ----- NSP Prediction -----
        nsp_pred = torch.argmax(nsp_logits, dim=1).item()  # 0 = NotNext, 1 = IsNext
        print(f"Sentence A: {sent_a}")
        print(f"Sentence B: {sent_b}")
        print("NSP Prediction:", "IsNext" if nsp_pred == 1 else "NotNext", "| True Label:", "IsNext" if is_next else "NotNext")

        # ----- MLM Prediction -----
        mlm_preds = torch.argmax(mlm_logits, dim=2).squeeze(1)
        mlm_tokens = []
        for idx, token in enumerate(tokens):
            if token == "[MASK]":
                pred_id = mlm_preds[idx].item()
                # Map back to word
                pred_word = [w for w, i in word2idx.items() if i == pred_id]
                mlm_tokens.append(pred_word[0] if pred_word else "[UNK]")
            else:
                mlm_tokens.append(token)

        print("MLM Prediction:", " ".join(mlm_tokens))
        print("-" * 50)

Sentence A: the cat sits on the mat
Sentence B: the mat is soft
NSP Prediction: IsNext | True Label: IsNext
MLM Prediction: [CLS] the cat sits on the mat [SEP] the bright is soft [SEP]
--------------------------------------------------
Sentence A: the dog runs in the park
Sentence B: birds fly in the sky
NSP Prediction: NotNext | True Label: NotNext
MLM Prediction: [CLS] the dog runs in the park [SEP] birds fly in the sky [SEP]
--------------------------------------------------
Sentence A: the sun rises in the east
Sentence B: it is very bright
NSP Prediction: IsNext | True Label: IsNext
MLM Prediction: [CLS] the sun rises in the east [SEP] it is very bright [SEP]
--------------------------------------------------
Sentence A: fish swim in the water
Sentence B: the moon shines at night
NSP Prediction: NotNext | True Label: NotNext
MLM Prediction: [CLS] fish swim in the water [SEP] the moon the at night [SEP]
--------------------------------------------------
